<a href="https://colab.research.google.com/github/naylanara89/ProyekAkhir_A_Avocado/blob/main/notebook/P12B_Varian1_kNN_Dasar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 12B — Varian 1: k-NN Dasar
## Dikerjakan oleh anggota 1

Pembelajaran Mesin INF62325 · S1 Informatika · UIN Siber Syekh Nurjati Cirebon

### Kedudukan varian ini

Varian 1 adalah **titik awal** yang dibandingkan oleh dua varian lain. Tugasnya bukan
menghasilkan angka tertinggi, melainkan menghasilkan angka pembanding yang jujur:
k-NN dengan parameter bawaan, tanpa penyetelan apa pun.

Varian ini juga memikul satu pembuktian yang dipakai seluruh kelompok: **bahwa
penskalaan memang wajib untuk model berbasis jarak.** Anda membuktikannya dengan
menjalankan k-NN yang sama dua kali, dengan dan tanpa penskalaan.

### Luaran yang dikumpulkan

| Luaran | Bentuk |
| --- | --- |
| Skor f1_macro lima lipatan | tercetak di notebook |
| Perbandingan dengan dan tanpa penskalaan | tabel tercetak di notebook |
| Biaya komputasi | tercetak di notebook |
| `varian1_knn_dasar.joblib` | berkas di folder proyek pada Drive |
| Jawaban tiga pertanyaan interpretasi | ditulis di sel markdown terakhir |

### Prasyarat

`dasar_bersama.joblib` sudah ada di folder proyek. Bila belum, jalankan notebook 12A.

---
## Langkah 0 — Samakan versi scikit-learn

Jalankan dua sel berikut setiap kali membuka notebook ini, sebelum sel apa pun
yang lain. Berkas model menyimpan rujukan ke kelas internal scikit-learn versi
tertentu, sehingga versi pelatihan dan versi pemuatan harus sama di seluruh
pertemuan. Angka 1.9.1 berlaku November 2026.

In [5]:
!pip install -q scikit-learn==1.9.1

**Wajib sekarang:** buka menu **Runtime**, pilih **Restart session**, lalu jalankan
ulang dari sel paling atas. Tanpa restart, Python masih memakai versi lama yang
sudah terlanjur dimuat ke memori.

In [6]:
import sklearn, pandas as pd, numpy as np, joblib

print('scikit-learn :', sklearn.__version__)
print('pandas       :', pd.__version__)

assert sklearn.__version__ == '1.9.1', (
    'Versi belum sesuai. Jalankan sel pip di atas, lalu '
    'Runtime -> Restart session, lalu ulangi dari sel pertama.'
)
print('Versi sudah sesuai.')

scikit-learn : 1.9.1
pandas       : 2.2.3
Versi sudah sesuai.


---
## Langkah 1 — Memuat dasar bersama

Notebook ini **tidak boleh membagi data sendiri**. Pembagian latih dan uji sudah
dibuat sekali di notebook 12A dan disimpan sebagai `dasar_bersama.joblib`. Bila
setiap anggota membagi datanya sendiri, ketiga varian tidak lagi dapat
dibandingkan, dan seluruh tabel perbandingan kelompok menjadi tidak sah.

Sel berikut juga mencetak **kunci data**, yaitu sidik ringkas dari pembagian yang
dipakai. Ketiga anggota harus memperoleh kunci yang sama persis.

In [7]:
import os, time
from google.colab import drive

drive.mount('/content/drive')
FOLDER = '/content/drive/MyDrive/proyek_akhir_ml'

NAMA_ANGGOTA = 'Nayla Rahma Farhanah'
NAMA_VARIAN = 'Varian 1 — k-NN dasar'

jalur = os.path.join(FOLDER, 'dasar_bersama.joblib')
if not os.path.exists(jalur):
    raise FileNotFoundError(
        f'{jalur} tidak ada. Satu orang di kelompok Anda harus menjalankan '
        'notebook 12A sampai selesai, lalu berkasnya dibagikan ke anggota lain.'
    )

d = joblib.load(jalur)
X_train, X_test = d['X_train'], d['X_test']
y_train, y_test = d['y_train'], d['y_test']
numerik, kategorik = d['numerik'], d['kategorik']
SEED = d['seed']
KUNCI_DATA = d['kunci_data']

print('Varian      :', NAMA_VARIAN)
print('Latih       :', X_train.shape, ' Uji:', X_test.shape)
print('Kunci data  :', KUNCI_DATA)
print('Tolok ukur f1_macro (model yang selalu menjawab no):', d['f1_tolok_ukur'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Varian      : Varian 1 — k-NN dasar
Latih       : (6400, 19)  Uji: (1600, 19)
Kunci data  : df3372b53e7c
Tolok ukur f1_macro (model yang selalu menjawab no): 0.4702


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


def buat_preprocessor(kolom_numerik, kolom_kategorik, pakai_skala):
    # pakai_skala=True untuk model berbasis jarak seperti k-NN.
    # pakai_skala=False untuk Naive Bayes yang tidak memakai jarak.
    langkah_numerik = [('imputasi', SimpleImputer(strategy='median'))]
    if pakai_skala:
        langkah_numerik.append(('skala', StandardScaler()))

    alur_kategorik = Pipeline([
        ('imputasi', SimpleImputer(strategy='most_frequent')),
        ('encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])

    return ColumnTransformer([
        ('num', Pipeline(langkah_numerik), kolom_numerik),
        ('kat', alur_kategorik, kolom_kategorik),
    ])


print('Fungsi buat_preprocessor siap dipakai.')

Fungsi buat_preprocessor siap dipakai.


---
## Langkah 2 — k-NN dengan parameter bawaan

Tiga parameter ditetapkan tetap, tanpa pencarian:

| Parameter | Nilai | Artinya |
| --- | --- | --- |
| `n_neighbors` | 5 | keputusan diambil dari 5 tetangga terdekat |
| `weights` | `uniform` | kelima tetangga punya suara sama besar |
| `metric` | `euclidean` | jarak diukur garis lurus antar titik |

Penskalaan dinyalakan, karena itu memang syarat model berbasis jarak. Pembuktiannya
ada di Langkah 3.

Yang diukur di sini hanya validasi silang pada data latih. **Data uji tidak dibuka di
notebook varian mana pun**, supaya pemilihan model nanti tidak dipengaruhi angka data
uji. Pembukaannya satu kali saja, di Pertemuan 13.

In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

parameter_varian = {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'euclidean'}

alur_v1 = Pipeline([
    ('prep', buat_preprocessor(numerik, kategorik, pakai_skala=True)),
    ('model', KNeighborsClassifier(**parameter_varian)),
])

mulai = time.perf_counter()
skor_varian = cross_val_score(alur_v1, X_train, y_train, cv=cv, scoring='f1_macro')
durasi = time.perf_counter() - mulai

print('Parameter        :', parameter_varian)
print('Skor tiap lipatan:', np.round(skor_varian, 4))
print('Rata-rata        :', round(float(skor_varian.mean()), 4))
print('Simpangan baku   :', round(float(skor_varian.std()), 4))
print('Waktu            :', round(durasi, 1), 'detik')

Parameter        : {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'euclidean'}
Skor tiap lipatan: [0.6503 0.6802 0.64   0.6623 0.6471]
Rata-rata        : 0.656
Simpangan baku   : 0.0141
Waktu            : 0.9 detik


---
## Langkah 3 — Membuktikan bahwa penskalaan wajib

Data ini punya kolom `age` yang bernilai puluhan dan kolom `nr.employed` yang bernilai
ribuan. Tanpa penskalaan, jarak antar titik hampir sepenuhnya ditentukan kolom yang
angkanya besar, sedangkan kolom lain nyaris tidak berpengaruh. Kalimat itu sering
diucapkan di kelas; sekarang Anda yang mengukurnya.

In [10]:
alur_tanpa_skala = Pipeline([
    ('prep', buat_preprocessor(numerik, kategorik, pakai_skala=False)),
    ('model', KNeighborsClassifier(**parameter_varian)),
])

skor_tanpa = cross_val_score(alur_tanpa_skala, X_train, y_train,
                             cv=cv, scoring='f1_macro')

tabel_skala = pd.DataFrame({
    'Praproses': ['Dengan penskalaan', 'Tanpa penskalaan'],
    'f1_macro rata-rata': [skor_varian.mean(), skor_tanpa.mean()],
    'Simpangan baku': [skor_varian.std(), skor_tanpa.std()],
}).round(4)
tabel_skala['Selisih'] = (tabel_skala['f1_macro rata-rata'] -
                          tabel_skala.loc[1, 'f1_macro rata-rata']).round(4)

print(tabel_skala.to_string(index=False))
print()
beda = skor_varian.mean() - skor_tanpa.mean()
print('Penskalaan menaikkan f1_macro sebesar', round(float(beda), 4))
if beda <= 0:
    print('Hasil ini di luar dugaan. Periksa ulang argumen pakai_skala di kedua alur.')

        Praproses  f1_macro rata-rata  Simpangan baku  Selisih
Dengan penskalaan               0.656          0.0141    0.019
 Tanpa penskalaan               0.637          0.0188    0.000

Penskalaan menaikkan f1_macro sebesar 0.019


Angka inilah yang dikutip dua anggota lain di laporan kelompok. Varian 2 menyetel
k-NN di atas praproses berskala, dan Varian 3 justru tidak memakai penskalaan sama
sekali. Keduanya perlu merujuk ke pengukuran Anda untuk menjelaskan pilihannya.

---
## Langkah 4 — Melatih model akhir varian ini

Validasi silang di atas hanya mengukur, tidak meninggalkan model yang bisa dipakai.
Sel berikut melatih sekali lagi pada seluruh data latih, dan itulah model yang
disimpan untuk Pertemuan 13.

In [11]:
alur_akhir = alur_v1
alur_akhir.fit(X_train, y_train)
print('Model Varian 1 terlatih pada', X_train.shape[0], 'baris.')

Model Varian 1 terlatih pada 6400 baris.


---
## Biaya komputasi varian ini

Waktu prediksi diukur pada 1.600 baris pertama data latih, bukan pada data uji.
Data uji sengaja belum dibuka sama sekali di notebook varian; pembukaannya satu
kali saja, di Pertemuan 13, setelah model terpilih ditetapkan. Ukuran berkas
model ikut dicatat karena layanan hosting gratis punya batas memori.

In [12]:
import tempfile

cuplik = X_train.head(1600)

mulai = time.perf_counter()
alur_akhir.predict(cuplik)
durasi_ms = (time.perf_counter() - mulai) * 1000

with tempfile.NamedTemporaryFile(suffix='.joblib', delete=False) as f:
    sementara = f.name
joblib.dump(alur_akhir, sementara)
ukuran_kb = os.path.getsize(sementara) / 1024
os.remove(sementara)

biaya_varian = {'prediksi_ms': round(durasi_ms, 1), 'ukuran_kb': round(ukuran_kb, 1)}
print('Waktu prediksi 1.600 baris :', biaya_varian['prediksi_ms'], 'ms')
print('Ukuran berkas model        :', biaya_varian['ukuran_kb'], 'KB')

Waktu prediksi 1.600 baris : 101.7 ms
Ukuran berkas model        : 3207.3 KB


---
## Langkah terakhir — Menyimpan hasil varian Anda

Berkas ini yang dibawa ke Pertemuan 13. Isinya model yang sudah terlatih, skor
tiap lipatan, biaya komputasi, nama Anda, dan kunci data. Kunci data itulah yang
diperiksa Pertemuan 13 untuk memastikan ketiga varian memakai pembagian yang sama.

In [13]:
if NAMA_ANGGOTA.startswith('andi'):
    raise ValueError('Isi dulu NAMA_ANGGOTA di Langkah 1.')

hasil = {
    'nama_varian': NAMA_VARIAN,
    'nama_tampil': 'Varian 1 — k-NN dasar',
    'nama_anggota': NAMA_ANGGOTA,
    'model': alur_akhir,
    'parameter': parameter_varian,
    'cv_lipatan': [round(float(v), 4) for v in skor_varian],
    'biaya': biaya_varian,
    'kunci_data': KUNCI_DATA,
    'versi_sklearn': sklearn.__version__,
}

jalur_hasil = os.path.join(FOLDER, 'varian1_knn_dasar.joblib')
joblib.dump(hasil, jalur_hasil)

print('Tersimpan :', jalur_hasil)
print('Ukuran    :', round(os.path.getsize(jalur_hasil) / 1024, 1), 'KB')
print('Pengerja  :', NAMA_ANGGOTA)
print('Rata-rata f1_macro :', round(float(np.mean(skor_varian)), 4))

Tersimpan : /content/drive/MyDrive/proyek_akhir_ml/varian1_knn_dasar.joblib
Ukuran    : 3207.6 KB
Pengerja  : Nayla Rahma Farhanah
Rata-rata f1_macro : 0.656


---
## Yang harus Anda tulis sendiri

Tiga pertanyaan berikut adalah bagian yang dinilai dari varian Anda. Jawablah dengan
angka yang baru saja Anda peroleh, bukan dengan kalimat umum.

**1. Berapa skor f1_macro varian Anda, dan berapa selisihnya dari tolok ukur yang
tercetak di Langkah 1?**

_(tulis jawaban di sini)_

**2. Berapa besar pengaruh penskalaan, dan mengapa pengaruhnya sebesar itu?** Sebut
setidaknya dua kolom yang skalanya jauh berbeda.

_(tulis jawaban di sini)_

**3. Apa yang belum dikerjakan varian ini, dan siapa yang mengerjakannya?** Jawaban
ini menunjukkan Anda memahami kedudukan varian Anda di dalam kelompok.

_(tulis jawaban di sini)_

---
## Daftar periksa sebelum notebook ini ditutup

 ✔ Kunci data yang tercetak sama dengan kunci dua anggota lain
 ✔ `NAMA_ANGGOTA` sudah diisi nama Anda
 ✔ Lima skor lipatan tercetak
 ✔ Tabel dengan dan tanpa penskalaan tercetak
 ✔ Berkas `varian1_knn_dasar.joblib` muncul di folder proyek
 ✔ Ketiga pertanyaan sudah dijawab, bukan masih berupa tanda kurung